# 실습 3. Boosting과 Stacking

## 데이터
- 분류: `sklearn.datasets.load_breast_cancer()`
- 회귀: `sklearn.datasets.load_diabetes()`

## 실습 목표
- GradientBoostingClassifier의 주요 하이퍼파라미터 비교
- HistGradientBoostingClassifier 학습과 평가
- StackingClassifier로 여러 모델의 예측 결합
- StackingRegressor로 회귀 모델 결합


In [6]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import StackingClassifier, StackingRegressor
from sklearn.metrics import classification_report, root_mean_squared_error, r2_score

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('분류 데이터:', X.shape, y.shape)
print('target names:', cancer.target_names)


분류 데이터: (569, 30) (569,)
target names: ['malignant' 'benign']


## 문제 1. GradientBoostingClassifier 기본 모델 학습

GradientBoostingClassifier를 학습하고 분류 성능을 확인하세요.

### 요구사항
- `n_estimators=100`
- `learning_rate=0.1`
- `max_depth=3`
- `random_state=42`
- 학습셋/평가셋 accuracy와 `classification_report()` 출력

### 실행 결과
```text
학습셋 accuracy: 약 1.0000
평가셋 accuracy: 약 0.9474 이상
classification_report가 출력됨
```


In [9]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)
gb_y_pred = gb_clf.predict(X_test)

print('학습셋 accuracy:', gb_clf.score(X_train, y_train))
print('평기셋 accuracy:', gb_clf.score(X_test, y_test))
print(classification_report(y_test, gb_y_pred))

학습셋 accuracy: 1.0
평기셋 accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## 문제 2. learning_rate와 n_estimators 비교

learning_rate와 n_estimators 조합별 점수를 비교하세요.

### 요구사항
- `learning_rate`: `[0.03, 0.1, 0.3]`
- `n_estimators`: `[50, 100, 200]`
- 결과 컬럼: `learning_rate`, `n_estimators`, `train_accuracy`, `test_accuracy`
- `test_accuracy` 기준 내림차순 정렬

### 실행 결과
```text
9개 조합의 결과 표가 출력됨
상위 조합은 test_accuracy가 약 0.95 이상으로 출력됨
```


In [10]:
param_grid = {
    'learning_rate': [0.03, 0.1, 0.3],
    'n_estimators': [50, 100, 200],
}

# GridSearchCV
grid_search = GridSearchCV(
    gb_clf,
    param_grid,
    scoring='accuracy',
    cv=3,
    return_train_score=True,
    n_jobs=1,
)

grid_search.fit(X_train, y_train)

grid_result_df = pd.DataFrame(grid_search.cv_results_)[[
    'param_learning_rate',
    'param_n_estimators',
    'mean_train_score',
    'mean_test_score',
]].rename(columns={
    'param_learning_rate': 'learning_rate',
    'param_n_estimators': 'n_estimators',
    'mean_train_score': 'train_accuracy',
    'mean_test_score': 'test_accuracy',
})

# rank가 1인 조합이 교차검증 기준 가장 좋은 조합임.
display(grid_result_df.sort_values('test_accuracy'))


,learning_rate,n_estimators,train_accuracy,test_accuracy
0,0.03,50,0.993407,0.931843
1,0.03,100,0.997803,0.938466
2,0.03,200,1.000000,0.942866
3,0.10,50,1.000000,0.945059
6,0.30,50,1.000000,0.945059
4,0.10,100,1.000000,0.956053
7,0.30,100,1.000000,0.958261
8,0.30,200,1.000000,0.958261
5,0.10,200,1.000000,0.960468


## 문제 3. HistGradientBoostingClassifier 학습

HistGradientBoostingClassifier를 학습하고 평가셋 accuracy를 확인하세요.

### 요구사항
- `max_iter=100`
- `learning_rate=0.1`
- `max_leaf_nodes=15`
- `random_state=42`
- 학습셋/평가셋 accuracy 출력

### 실행 결과
```text
학습셋 accuracy와 평가셋 accuracy가 출력됨
평가셋 accuracy는 약 0.94 이상으로 출력됨
```


In [11]:
hist_clf = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=15,
    random_state=42
)
hist_clf.fit(X_train, y_train)
hist_y_pred = hist_clf.predict(X_test)

print('학습셋 accuracy:', hist_clf.score(X_train, y_train))
print('평기셋 accuracy:', hist_clf.score(X_test, y_test))
print(classification_report(y_test, hist_y_pred))

학습셋 accuracy: 1.0
평기셋 accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## 문제 4. StackingClassifier 구성과 비교

Logistic Regression, KNN, Decision Tree를 기본 모델로 사용해 StackingClassifier를 구성하세요.

### 요구사항
- Logistic Regression과 KNN은 `Pipeline([('scaler', StandardScaler()), ...])`로 구성
- Decision Tree는 `max_depth=4`, `random_state=42`
- final estimator는 `LogisticRegression(max_iter=1000)` 사용
- 개별 모델과 Stacking 모델의 train/test accuracy를 표로 비교

### 실행 결과
```text
logistic_regression, knn, decision_tree, stacking 4개 모델의 점수 표가 출력됨
```


In [12]:
base_regressors = [
    ('logistic_regression',
     Pipeline([('scaler', StandardScaler()),
               ('model', LogisticRegression())])),

    ('knn',
     Pipeline([('scaler', StandardScaler()),
               ('model', KNeighborsRegressor(n_neighbors=5))])),

    ('decision_tree_regressor',
     DecisionTreeRegressor(max_depth=4, random_state=42))
]

stacking_reg = StackingRegressor(
    estimators = base_regressors,    # 기본 모델
    final_estimator = LogisticRegression(max_iter=1000),      # 최종 모델
    cv = 5,     # 교차 검증 5회
    n_jobs = 1  # 벙렬 처리 X
)

regression_models = dict(base_regressors)
regression_models['stacking_regressor'] = stacking_reg

regression_results = []
for name, model in regression_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    regression_results.append({
        'model': name,
        # R2는 1에 가까울수록 평균 예측보다 모델 설명력이 좋다는 뜻임.
        'test_R2': r2_score(y_test, pred),
        # RMSE는 예측 오차의 평균적인 크기이며 낮을수록 좋음.
        'test_RMSE': root_mean_squared_error(y_test, pred)
    })

regression_result_df = pd.DataFrame(regression_results).sort_values('test_RMSE')
display(regression_result_df)

ValueError: The estimator Pipeline should be a regressor.

## 문제 5. StackingRegressor 회귀 모델 학습

Diabetes 데이터로 StackingRegressor를 만들고 회귀 지표를 비교하세요.

### 요구사항
- 기본 모델: Ridge, KNeighborsRegressor, DecisionTreeRegressor
- Ridge와 KNN은 스케일링 Pipeline 사용
- final estimator는 `Ridge()` 사용
- 평가셋 R2와 RMSE를 표로 출력

### 실행 결과
```text
ridge, knn_regressor, decision_tree_regressor, stacking_regressor 4개 모델의 회귀 지표가 출력됨
```


In [13]:
diabetes = load_diabetes(as_frame=True)
reg_X = diabetes.data
reg_y = diabetes.target

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42
)